# Reaction diagrams from an rxflow database

Set `DB_PATH` below to an HPC results database. A profile is plotted only after its jobs are no longer pending/running and a successful TS plus retained reactant and product minima exist. Each overlay shows all completed binding permutations for a ligand on one shared Gibbs-energy scale. Then the notebook renders the XYZ structures for the completed profile with the lowest reactant Gibbs energy. Endpoint structures are the lowest-Gibbs retained conformers; the TS structure is its completed optimized geometry. Energies are from ORCA NumFreq and shown in kcal/mol.

In [ ]:
from pathlib import Path
import sqlite3
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import xyzrender
from IPython.display import display, Markdown

DB_PATH = Path("hpc_results/workflow.sqlite")  # change to your database
HARTREE_TO_KCAL = 627.509474

if not DB_PATH.is_file():
    raise FileNotFoundError(f"Database not found: {DB_PATH.resolve()}")
with sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True) as connection:
    cursor = connection.execute("""
        SELECT ligand_id, pair_index, seating, endpoint, conformer, status, retained,
               gibbs_free_energy_eh, binding_permutation, binding_mode,
               coordination_isomer, substrate_site, xyz_file, donor, hydrogen, acceptor
        FROM jobs
    """)
    columns = [column[0] for column in cursor.description]
    jobs = [dict(zip(columns, row)) for row in cursor.fetchall()]
print(f"Loaded {len(jobs)} job rows from {DB_PATH}")
counts = Counter((row["endpoint"], row["status"]) for row in jobs)
display([{ "endpoint": endpoint, "status": status, "rows": count }
         for (endpoint, status), count in sorted(counts.items())])

A profile requires at least one successful, retained reactant and product conformer with a Gibbs energy, plus a successful TS row with a Gibbs energy. Any pending/running job for that pair/seating keeps the profile out until the calculations finish. The lowest-Gibbs retained conformer is selected separately for reactant and product. Each colored pathway is labeled by binding permutation, pair, seating and substrate site.

In [ ]:
# Keep successful results with Gibbs energies; failed or still-running rows are excluded.
keys = ("ligand_id", "pair_index", "seating", "binding_permutation")
groups = defaultdict(list)
for row in jobs:
    key = tuple(row[column] for column in keys)
    groups[key].append(row)

profiles = []
for key, all_rows in groups.items():
    if any(row["status"] in ("pending", "running") for row in all_rows):
        continue
    rows = [row for row in all_rows if row["status"] == "succeeded"
            and row["gibbs_free_energy_eh"] is not None]
    reactants = [r for r in rows if r["endpoint"] == "reactant" and r["retained"] == 1]
    products = [r for r in rows if r["endpoint"] == "product" and r["retained"] == 1]
    transition_states = [r for r in rows if r["endpoint"] == "TS"]
    if not (reactants and products and transition_states):
        continue
    reactant = min(reactants, key=lambda row: row["gibbs_free_energy_eh"])
    product = min(products, key=lambda row: row["gibbs_free_energy_eh"])
    ts = min(transition_states, key=lambda row: row["gibbs_free_energy_eh"])
    profiles.append(dict(zip(keys, key), reactant=reactant["gibbs_free_energy_eh"],
                         TS=ts["gibbs_free_energy_eh"], product=product["gibbs_free_energy_eh"],
                         reactant_row=reactant, product_row=product, ts_row=ts,
                         binding_mode=reactant["binding_mode"],
                         coordination_isomer=reactant["coordination_isomer"],
                         substrate_site=reactant["substrate_site"]))
print(f"Complete reaction profiles: {len(profiles)}")
print(f"Incomplete groups excluded: {len(groups) - len(profiles)}")
display(sorted(profiles, key=lambda row: (str(row["ligand_id"]), row["binding_permutation"] or 0,
                                          row["pair_index"], row["seating"]))[:20])

In [ ]:
if not profiles:
    print("No complete reactant–TS–product profiles yet.")
else:
    import matplotlib as mpl
    x = [0, 1, 2]
    labels = ["Reactant", "TS", "Product"]
    half_width = 0.24
    ligand_ids = dict.fromkeys(row["ligand_id"] for row in profiles)

    for ligand_id in ligand_ids:
        group = [row for row in profiles if row["ligand_id"] == ligand_id]
        reference = min(row["reactant"] for row in group)
        permutations = dict.fromkeys(row["binding_permutation"] for row in group)
        colors = {permutation: mpl.colormaps["tab20"](i % 20)
                  for i, permutation in enumerate(permutations)}
        fig, ax = plt.subplots(figsize=(10, 6))

        for index, row in enumerate(group):
            energies = [row["reactant"], row["TS"], row["product"]]
            relative = [(energy - reference) * HARTREE_TO_KCAL for energy in energies]
            color = colors[row["binding_permutation"]]
            site = "?" if row["substrate_site"] is None else row["substrate_site"]
            label = (f"perm {row['binding_permutation']} (mode {row['binding_mode']}, "
                     f"isomer {row['coordination_isomer']}, site {site}, pair {row['pair_index']})")

            for state, (position, energy) in enumerate(zip(x, relative)):
                ax.hlines(energy, position - half_width, position + half_width,
                          color=color, linewidth=3.5, zorder=3)
                ax.scatter(position, energy, color="#D62728" if state == 1 else color,
                           edgecolor="white", linewidth=0.6, s=38, zorder=4,
                           marker="D" if state == 1 else "o",
                           label=label if state == 0 else None)
                if len(group) <= 6:
                    ax.annotate(f"{energy:+.1f}", xy=(position, energy), xytext=(0, 8),
                                textcoords="offset points", ha="center", fontsize=8, color=color)
            for state in range(2):
                ax.plot([x[state] + half_width, x[state + 1] - half_width],
                        [relative[state], relative[state + 1]],
                        color="#666666", linewidth=1.4, zorder=2)

        ax.axhline(0, color="black", linestyle=":", linewidth=1, alpha=0.6)
        ax.set_xticks(x, labels)
        ax.set_xlim(-0.55, 2.55)
        ax.set_ylabel(r"Relative Gibbs free energy, $\Delta G$ (kcal/mol)", fontsize=12)
        ax.set_xlabel("Reaction coordinate", fontsize=12)
        ax.set_title(f"Ligand {ligand_id}: Gibbs reaction profiles", fontsize=14)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.grid(axis="y", alpha=0.2)
        ax.legend(title="Binding permutation", fontsize=8, bbox_to_anchor=(1.02, 1),
                  loc="upper left")
        fig.tight_layout()
        plt.show()

        # Render the three structures from the profile with the lowest reactant Gibbs energy.
        lowest = min(group, key=lambda row: row["reactant"])
        display(Markdown(
            f"### Ligand {ligand_id}: lowest-reactant profile "
            f"(permutation {lowest['binding_permutation']}, pair {lowest['pair_index']}, "
            f"seating {lowest['seating']}, site {lowest['substrate_site']})"
        ))
        for title, row_key in (("Reactant", "reactant_row"), ("Transition state", "ts_row"),
                               ("Product", "product_row")):
            structure = lowest[row_key]
            xyz_path = DB_PATH.resolve().parent / structure["xyz_file"]
            display(Markdown(f"**{title}** — $G = {structure['gibbs_free_energy_eh']:.8f}$ Eh"))
            if not xyz_path.is_file():
                print(f"XYZ not found: {xyz_path}")
                continue
            render_options = {"no_hy": True, "bo": False}
            if row_key == "ts_row":
                reacting_atoms = (structure["donor"], structure["hydrogen"], structure["acceptor"])
                if all(atom is not None for atom in reacting_atoms):
                    donor, hydrogen, acceptor = (atom + 1 for atom in reacting_atoms)
                    render_options["ts_bonds"] = [(donor, hydrogen), (hydrogen, acceptor)]
            display(xyzrender.render(xyz_path, **render_options))